In [30]:
# Veri setini çekelim
import pandas as pd
url = "https://storage.googleapis.com/download.tensorflow.org/data/heart.csv"
df = pd.read_csv(url)
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,1,145,233,1,2,150,0,2.3,3,0,fixed,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3,normal,1
2,67,1,4,120,229,0,2,129,1,2.6,2,2,reversible,0
3,37,1,3,130,250,0,0,187,0,3.5,3,0,normal,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0,normal,0


In [31]:
# Güzel, thal sütunu dışındakiler sayısala çevrilmiş gözüküyor. Eksik veriye bakalım:
df.isna().sum()

,0
age,0
sex,0
cp,0
trestbps,0
chol,0
fbs,0
restecg,0
thalach,0
exang,0
oldpeak,0


In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        303 non-null    int64  
 12  thal      303 non-null    object 
 13  target    303 non-null    int64  
dtypes: float64(1), int64(12), object(1)
memory usage: 33.3+ KB


In [33]:
df['thal'].value_counts()

,count
thal,
normal,168
reversible,115
fixed,18
1,1
2,1


In [34]:
# that içinde anlamsız 1 ve 2 yazan değerler var, 1 er adet. Bunları silebiliriz.
df = df[~df['thal'].isin(['1', '2'])]
df['thal'].value_counts()

,count
thal,
normal,168
reversible,115
fixed,18


In [35]:
# that sütunu object, get dummies yapalım
df = pd.get_dummies(df, columns=['thal'], drop_first=True)
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,target,thal_normal,thal_reversible
0,63,1,1,145,233,1,2,150,0,2.3,3,0,0,False,False
1,67,1,4,160,286,0,2,108,1,1.5,2,3,1,True,False
2,67,1,4,120,229,0,2,129,1,2.6,2,2,0,False,True
3,37,1,3,130,250,0,0,187,0,3.5,3,0,0,True,False
4,41,0,2,130,204,0,2,172,0,1.4,1,0,0,True,False


In [36]:
df.isnull().sum()

,0
age,0
sex,0
cp,0
trestbps,0
chol,0
fbs,0
restecg,0
thalach,0
exang,0
oldpeak,0


In [37]:
# Şimdi train test split e geçebiliriz
from sklearn.model_selection import train_test_split
x = df.drop('target', axis=1)
y = df['target']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [38]:
# Standardscaler yapalım
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_train[['age', 'trestbps', 'chol', 'thalach', 'oldpeak']] = scaler.fit_transform(x_train[['age', 'trestbps', 'chol', 'thalach', 'oldpeak']])
x_test[['age', 'trestbps', 'chol', 'thalach', 'oldpeak']] = scaler.transform(x_test[['age', 'trestbps', 'chol', 'thalach', 'oldpeak']])

In [39]:
# şimdi logistic regression modelimizi kuralım
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(class_weight='balanced')
model.fit(x_train, y_train)

LogisticRegression(class_weight='balanced')

In [40]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

tahmin = model.predict(x_test)

print(accuracy_score(y_test, tahmin))
print(classification_report(y_test, tahmin))
print(confusion_matrix(y_test, tahmin))

0.8524590163934426
              precision    recall  f1-score   support

           0       0.93      0.87      0.90        47
           1       0.65      0.79      0.71        14

    accuracy                           0.85        61
   macro avg       0.79      0.83      0.81        61
weighted avg       0.87      0.85      0.86        61

[[41  6]
 [ 3 11]]


In [41]:
# RandomForest ile deneyelim
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(class_weight='balanced', random_state=42)
model_rf.fit(x_train, y_train)

tahmin_rf = model_rf.predict(x_test)

print(accuracy_score(y_test, tahmin_rf))
print(classification_report(y_test, tahmin_rf))
print(confusion_matrix(y_test, tahmin_rf))

0.8032786885245902
              precision    recall  f1-score   support

           0       0.86      0.89      0.88        47
           1       0.58      0.50      0.54        14

    accuracy                           0.80        61
   macro avg       0.72      0.70      0.71        61
weighted avg       0.79      0.80      0.80        61

[[42  5]
 [ 7  7]]


In [42]:
# SVC modelini deneyelim
from sklearn.svm import SVC

model_svc = SVC(class_weight='balanced', random_state=42)
model_svc.fit(x_train, y_train)

tahmin_svc = model_svc.predict(x_test)

print(accuracy_score(y_test, tahmin_svc))
print(classification_report(y_test, tahmin_svc))
print(confusion_matrix(y_test, tahmin_svc))

0.8524590163934426
              precision    recall  f1-score   support

           0       0.93      0.87      0.90        47
           1       0.65      0.79      0.71        14

    accuracy                           0.85        61
   macro avg       0.79      0.83      0.81        61
weighted avg       0.87      0.85      0.86        61

[[41  6]
 [ 3 11]]


In [48]:
# Logistic Regression ve SVC iyi sonuç verdi. Logistic modelini saklayabiliriz
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
pipe = make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=1000))
cross_val_score(pipe, x, y, cv=5, scoring='recall')

array([1.        , 0.9375    , 0.6875    , 0.76470588, 0.82352941])

In [49]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

final_pipe = make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=1000))
final_pipe.fit(x, y)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [50]:
# pickle yapıp modeli saklayalım
import pickle
pickle.dump(final_pipe, open(r"heart_model.pkl", "wb"))